# Đánh giá hiệu năng bộ phân tích cú pháp

Chương này trình bày các phép đo hiệu năng thực tế của thuật toán xác định đường biên dịch, bóc tách cây phân cấp và khử trùng lặp nội dung.

Bảng và biểu đồ hiệu năng trong chương này sinh trực tiếp khi build từ tệp `report_book/data/m2_benchmark.json`. Các nhận định định lượng dùng ký hiệu `[...]` cho đến khi có log đo đầy đủ trên Google Colab ở chế độ CPU.

## 1. Môi trường thực nghiệm và tổng hợp chỉ số

Bài kiểm tra hiệu năng cho bộ phân tích cú pháp chạy trên cùng cấu hình Google Colab ở chế độ CPU: 2 vCPU Intel Xeon 2.20 GHz, 12.67 GB RAM, ổ đĩa 107.72 GB.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("../data/m2_benchmark.json", "r", encoding="utf-8") as f:
    m2_bench = json.load(f)

t_m2 = m2_bench["time_metrics"]
m_m2 = m2_bench["memory_metrics"]
s_m2 = m2_bench["storage_metrics"]

p_bench_df = pd.DataFrame([
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Tổng thời gian phân tích cú pháp toàn bộ", "Giá trị": f"{t_m2['total_wall_time_seconds'] / 3600:.2f} giờ ({t_m2['total_wall_time_seconds']:.1f} s)"},
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Thời gian trung bình mỗi bài báo", "Giá trị": f"{t_m2['avg_time_per_publication_seconds']:.2f} giây"},
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Thời gian trung bình mỗi phiên bản", "Giá trị": f"{t_m2['avg_time_per_version_seconds']:.2f} giây"},
    {"Nhóm chỉ số": "Bộ nhớ", "Chỉ số đo": "Mức RAM đỉnh", "Giá trị": f"{m_m2['peak_ram_mb']:.1f} MB"},
    {"Nhóm chỉ số": "Bộ nhớ", "Chỉ số đo": "Mức RAM trung bình theo mẫu", "Giá trị": f"{m_m2['avg_ram_mb']:.1f} MB"},
    {"Nhóm chỉ số": "Lưu trữ", "Chỉ số đo": "Dung lượng đĩa tối đa khi xử lý", "Giá trị": f"{s_m2['peak_disk_usage_gb']:.2f} GB"},
    {"Nhóm chỉ số": "Lưu trữ", "Chỉ số đo": "Dung lượng dữ liệu Parquet xuất bản", "Giá trị": f"{s_m2['final_parquet_output_gb']:.2f} GB"}
])

p_bench_df.style.hide(axis="index").set_caption("Bảng 2.2: Các chỉ số hiệu năng của bộ phân tích cú pháp trên Google Colab ở chế độ CPU")


## 2. Tốc độ xử lý và thời gian

Khác với việc tải dữ liệu vốn chờ mạng, khâu parsing tiêu tốn năng lực xử lý của CPU để so khớp biểu thức chính quy và dựng cây trong bộ nhớ.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

t_labels = ["Mỗi bài báo", "Mỗi phiên bản"]
t_vals = [t_m2["avg_time_per_publication_seconds"], t_m2["avg_time_per_version_seconds"]]
ax1.bar(t_labels, t_vals, color=["#41b6c4", "#253494"], width=0.45, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Giây", fontsize=10)
ax1.set_title("Hình 2.4a: Thời gian xử lý trung bình mỗi đơn vị", fontsize=11)
for i, v in enumerate(t_vals):
    ax1.text(i, v + 0.03, f"{v:.2f} s", ha="center", va="bottom", fontsize=10)
ax1.set_ylim(0, max(t_vals) * 1.25)

r_labels = ["RAM đỉnh", "RAM trung bình"]
r_vals = [m_m2["peak_ram_mb"], m_m2["avg_ram_mb"]]
ax2.bar(r_labels, r_vals, color=["#fc4e2a", "#feb24c"], width=0.45, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Dung lượng (MB)", fontsize=10)
ax2.set_title("Hình 2.4b: Mức tiêu thụ bộ nhớ RAM của bộ phân tích cú pháp", fontsize=11)
for i, v in enumerate(r_vals):
    ax2.text(i, v + 20, f"{v:.1f} MB", ha="center", va="bottom", fontsize=10)
ax2.set_ylim(0, max(r_vals) * 1.25)

plt.tight_layout()
plt.show()


### Nhận xét về tốc độ và bộ nhớ

- Tốc độ xử lý trung bình đạt `[...]` giây cho mỗi bài báo (khoảng `[...]` giây cho mỗi phiên bản). Với quy mô tập dữ liệu, toàn bộ quá trình parsing mất khoảng `[...]` giờ trên 2 vCPU của Colab.
- Mức RAM đỉnh là `[...] MB` và mức trung bình là `[...] MB`. Nhóm giải phóng bộ nhớ của từng cây sau khi ghi dữ liệu trung gian, tránh tích tụ bộ nhớ khi chạy lâu.


## 3. Hiệu quả nén và dung lượng lưu trữ

Dữ liệu đầu ra của Giai đoạn 2 được lưu thành các tệp Parquet kèm nén Snappy. Đồ thị dưới đây so sánh dung lượng giữa tập TeX thô đầu vào và kích thước Parquet sau khi cấu trúc hóa và khử trùng lặp.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
s_cats = ["Dữ liệu TeX và Bib thô", "Tập dữ liệu Parquet"]
s_vals = [6.20, s_m2["final_parquet_output_gb"]]
colors = ["#756bb1", "#31a354"]

bars = ax.bar(s_cats, s_vals, color=colors, width=0.45, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Dung lượng lưu trữ (GB)", fontsize=10)
ax.set_title("Hình 2.5: So sánh dung lượng dữ liệu thô và tập dữ liệu Parquet", fontsize=12, pad=12)

for bar, val in zip(bars, s_vals):
    ax.text(bar.get_x() + bar.get_width()/2.0, val + 0.15, f"{val:.2f} GB", ha="center", va="bottom", fontsize=10)

reduction_s = (1 - s_vals[1] / s_vals[0]) * 100
ax.text(0.5, max(s_vals) * 0.7, f"Giảm: {reduction_s:.1f}%", fontsize=11, ha="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#e5f5e0", edgecolor="#a1d99b"))

ax.set_ylim(0, max(s_vals) * 1.25)
plt.tight_layout()
plt.show()


### Nhận xét về dung lượng lưu trữ

- Kích thước tập dữ liệu có cấu trúc đầu ra chiếm `[...] GB`, giảm `[...]%` so với dữ liệu văn bản thô ban đầu.
- Mức giảm này đến từ cơ chế khử trùng lặp qua các phiên bản và khả năng nén theo cột của định dạng Parquet đối với các trường văn bản có tính lặp lại.


## 4. Tổng kết chuẩn bị cho Giai đoạn 3

Kết quả thực nghiệm cho thấy:

- Dữ liệu của ba bảng `elements`, `hierarchy` và `bibtex` đã được chuẩn bị đầy đủ để làm đầu vào cho bài toán học máy ở Giai đoạn 3.
- Bảng `bibtex` sau khi chuẩn hóa cung cấp danh sách các mục trích dẫn đã phân rã thành các trường tác giả, tiêu đề và năm xuất bản. Đây là tập cần nối với các ứng viên trong bảng `references` từ Giai đoạn 1.
- Cấu trúc phân cấp trong bảng `hierarchy` giúp nhóm có thể trích chọn thêm các đặc trưng cấu trúc cho mô hình phân loại ở Giai đoạn 3.